# Meta x Hugging Face Hackathon: BESCOM EV Grid Oracle
### RLHF Training Pipeline with Unsloth & GRPO

This notebook implements the training pipeline for an LLM-based Grid Oracle using **Group Relative Policy Optimization (GRPO)**. It is optimized for the **Google Colab T4 GPU**.

**Author:** Elite AI Researcher Pipeline

## 1. Setup & Dependencies
We use Unsloth for fast 4-bit quantization and TRL for the GRPOTrainer.

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install gymnasium wandb datasets

## 2. Environment Implementation
We define the `BESCOM_EV_Env` directly in the notebook for easier debugging.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class BESCOM_EV_Env(gym.Env):
    def __init__(self):
        super(BESCOM_EV_Env, self).__init__()
        self.action_space = spaces.Discrete(5)
        self.observation_space = spaces.Text(max_length=1000)
        self.reset()
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = {
            "transformer_load": np.random.uniform(0.5, 0.9),
            "ev_queue_length": np.random.randint(5, 20),
            "peak_hours": np.random.choice([True, False])
        }
        return self._get_obs(), {}

    def _get_obs(self):
        return f"Grid Status: Load={self.state['transformer_load']:.2f}, Queue={self.state['ev_queue_length']}, Peak={self.state['peak_hours']}"

    def step(self, action):
        reward = 0.0
        terminated = False
        if action == 3: # schedule
            reward = 0.5
            self.state["transformer_load"] += 0.1
        elif action == 4: # report
            reward = 1.0
            terminated = True
        if self.state["transformer_load"] > 0.95:
            reward -= 2.0
            terminated = True
        return self._get_obs(), reward, terminated, False, {}

## 3. GRPO Training Script
Running the core training logic.

In [ ]:
# Import the logic from our training script or paste here
!python train_grpo.py

## 4. Final Save
Ensure the model is saved in FP16 format for submission.

In [ ]:
print("Model saved as 'trained_ev_oracle' folder. Download this for your submission!")